# Challenge 2: Online Purchase Prediction (Team 11)

**Goal:** predict whether an online shopping session ends in a purchase (`Revenue = True`).

**Primary metric:** F1 score for the positive class (`Revenue = True`). The target is imbalanced, so accuracy alone is misleading.

**Rules followed:** only `challenge2_train.csv` is used for exploration, feature engineering, validation, tuning, and threshold selection. `challenge2_test.csv` is used once, at the end, to write `predictions.csv`. `ID` is never a predictor.

## 1. Setup and data loading

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (GradientBoostingClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score, classification_report,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
DATA = Path("data")

train = pd.read_csv(DATA / "challenge2_train.csv")
test = pd.read_csv(DATA / "challenge2_test.csv")
print("train:", train.shape, "| test:", test.shape)
assert train["ID"].is_unique and test["ID"].is_unique
assert not set(train["ID"]) & set(test["ID"])

## 2. Exploratory data analysis

In [ ]:
y = train["Revenue"].astype(bool)
print(y.value_counts().rename({True: "Purchase (True)", False: "No purchase (False)"}))
print(f"Positive rate: {y.mean():.2%}")
print("Missing values in train:", int(train.isna().sum().sum()), "| test:", int(test.isna().sum().sum()))
train.describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y.value_counts().plot.bar(ax=axes[0], color=["#98A2B3", "#1570EF"])
axes[0].set_title("Target balance"); axes[0].set_xlabel("Revenue"); axes[0].set_ylabel("Sessions")

rate_pv = train.groupby(train["PageValues"] > 0)["Revenue"].mean()
rate_pv.index = ["PageValues = 0", "PageValues > 0"]
rate_pv.plot.bar(ax=axes[1], color="#1570EF")
axes[1].set_title("Purchase rate by PageValues"); axes[1].set_ylabel("Purchase rate")

month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
train.groupby("Month")["Revenue"].mean().reindex([m for m in month_order if m in set(train["Month"])]).plot.bar(
    ax=axes[2], color="#1570EF")
axes[2].set_title("Purchase rate by month"); axes[2].set_ylabel("Purchase rate")
for ax in axes[1:]:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

print(train.groupby("VisitorType")["Revenue"].agg(["count", "mean"]).round(3))

**EDA takeaways**

- 1,431 of 9,247 training sessions (15.48%) end in a purchase. A model that always predicts "no purchase" is 84.5% accurate but has F1 = 0, so positive-class F1 is the right primary metric.
- `PageValues` separates the classes strongly: 56.9% of sessions with `PageValues > 0` purchase, versus 3.6% of sessions with no page value.
- Purchase rates vary by month, from 1.4% in February to 22.4% in October and 24.8% in November. They also vary by visitor type: new visitors purchase in 24.7% of sessions versus 14.0% for returning visitors.
- Page-count and duration features are heavily right-skewed (for example, `ProductRelated_Duration` has a median of 598 s but a maximum of 63,974 s), so they are log-transformed before scaling.
- There are no missing values in either dataset.

## 3. Preprocessing and feature engineering

In [ ]:
CATEGORICAL = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
SKEWED = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
          "ProductRelated", "ProductRelated_Duration", "PageValues"]


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    # Row-wise features only, so nothing is learned from other rows (no leakage).
    out = df.drop(columns=["ID", "Revenue"], errors="ignore").copy()
    out["Weekend"] = out["Weekend"].astype(int)
    out["TotalPages"] = out[["Administrative", "Informational", "ProductRelated"]].sum(axis=1)
    out["TotalDuration"] = out[["Administrative_Duration", "Informational_Duration",
                                "ProductRelated_Duration"]].sum(axis=1)
    out["HasPageValue"] = (out["PageValues"] > 0).astype(int)
    per_page = out["ProductRelated_Duration"] / out["ProductRelated"].replace(0, np.nan)
    out["ProductDurationPerPage"] = per_page.fillna(0)
    out["IsQ4"] = out["Month"].isin(["Oct", "Nov", "Dec"]).astype(int)
    for col in CATEGORICAL:
        out[col] = out[col].astype(str)
    return out


X = engineer(train)
X_test = engineer(test)
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]
LOG_COLS = SKEWED + ["TotalPages", "TotalDuration", "ProductDurationPerPage"]
OTHER_NUMERIC = [c for c in NUMERIC if c not in LOG_COLS]
print("Features:", X.shape[1], "| numeric:", len(NUMERIC), "| categorical:", len(CATEGORICAL))


def make_preprocessor(scale: bool = True) -> ColumnTransformer:
    # Fitted inside each CV fold through the Pipeline, so validation rows never influence it.
    log_steps = [("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))]
    if scale:
        log_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("log_numeric", Pipeline(log_steps), LOG_COLS),
        ("numeric", StandardScaler() if scale else "passthrough", OTHER_NUMERIC),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])

## 4. Validation strategy

- **Stratified 5-fold cross-validation** on the training set only, with a fixed seed. Stratification keeps the purchase rate the same in every fold.
- All preprocessing lives inside a scikit-learn `Pipeline`, so it is re-fitted on each training fold; validation folds never influence scaling or encoding.
- **Primary metric:** F1 for `Revenue = True`. We also report precision, recall, ROC-AUC, and average precision, plus train-fold F1 to detect overfitting.
- Grid Search uses the same folds and F1 scoring. The final decision threshold is chosen from out-of-fold training predictions, never from the test set.
- The test set is used only once, in Section 9.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {"f1": "f1", "precision": "precision", "recall": "recall",
           "roc_auc": "roc_auc", "avg_precision": "average_precision"}


def evaluate(name, model, scale=True):
    pipe = Pipeline([("prep", make_preprocessor(scale)), ("model", model)])
    res = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1)
    row = {"model": name}
    for key in SCORING:
        row[f"cv_{key}"] = res[f"test_{key}"].mean()
    row["cv_f1_std"] = res["test_f1"].std()
    row["train_f1"] = res["train_f1"].mean()
    row["overfit_gap"] = row["train_f1"] - row["cv_f1"]
    row["fit_seconds"] = res["fit_time"].mean()
    return row

## 5. Baseline comparison of the seven required models

In [ ]:
baseline_models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED), True),
    "Naive Bayes (Gaussian)": (GaussianNB(), True),
    "K-Nearest Neighbors": (KNeighborsClassifier(n_neighbors=15), True),
    "Support Vector Machine": (SVC(kernel="rbf", class_weight="balanced", random_state=SEED), True),
    "Decision Tree": (DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=SEED), False),
    "Random Forest": (RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                             random_state=SEED, n_jobs=-1), False),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=SEED), False),
}
baseline = pd.DataFrame([evaluate(n, m, s) for n, (m, s) in baseline_models.items()])
baseline = baseline.sort_values("cv_f1", ascending=False).reset_index(drop=True)
baseline.round(3)

In [ ]:
ordered = baseline.set_index("model").sort_values("cv_f1")
ax = ordered["cv_f1"].plot.barh(xerr=ordered["cv_f1_std"], color="#1570EF", figsize=(8, 4))
ax.set_xlabel("Mean 5-fold CV F1 (Revenue = True)"); ax.set_title("Baseline models")
plt.tight_layout(); plt.show()

**Baseline interpretation (stratified 5-fold CV, untuned settings)**

- Gradient Boosting (0.664), SVM (0.663), Random Forest (0.660), and Logistic Regression (0.658) are close on F1. Their ROC-AUC values are all 0.92–0.93.
- The models reach similar F1 in different ways. The class-weighted SVM and Logistic Regression favor recall (~0.83–0.85) at lower precision (~0.55). The tree ensembles favor precision (0.71–0.75) at lower recall (~0.59–0.62).
- The untuned Random Forest overfits heavily: train F1 = 1.00 versus CV F1 = 0.66.
- Gaussian Naive Bayes is clearly the weakest (F1 = 0.30). It flags almost everything as a purchase (recall 0.98, precision 0.18) because its independence assumption fails on the strongly correlated page and duration features.

## 6. Improving models with Grid Search

In [ ]:
def grid(name, model, params, scale=True):
    pipe = Pipeline([("prep", make_preprocessor(scale)), ("model", model)])
    search = GridSearchCV(pipe, {f"model__{k}": v for k, v in params.items()}, scoring="f1",
                          cv=CV, n_jobs=-1, return_train_score=True)
    search.fit(X, y)
    train_f1 = search.cv_results_["mean_train_score"][search.best_index_]
    print(f"{name}: best CV F1 = {search.best_score_:.4f} | train F1 = {train_f1:.4f} | {search.best_params_}")
    return search


searches = {
    "Logistic Regression": grid("Logistic Regression",
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
        {"C": [0.01, 0.1, 1, 10]}),
    "K-Nearest Neighbors": grid("K-Nearest Neighbors", KNeighborsClassifier(),
        {"n_neighbors": [5, 15, 31], "weights": ["uniform", "distance"]}),
    "Support Vector Machine": grid("Support Vector Machine",
        SVC(kernel="rbf", class_weight="balanced", random_state=SEED),
        {"C": [0.3, 1, 3], "gamma": ["scale", 0.01]}),
    "Random Forest": grid("Random Forest",
        RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                               random_state=SEED, n_jobs=-1),
        {"max_depth": [8, 12, None], "min_samples_leaf": [1, 5, 10]}, scale=False),
    "Gradient Boosting (HistGB)": grid("Gradient Boosting (HistGB)",
        HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
        {"learning_rate": [0.03, 0.1], "max_depth": [3, 6], "max_iter": [200, 400]}, scale=False),
}

In [ ]:
tuned = pd.DataFrame([
    {"model": name, "tuned_cv_f1": s.best_score_,
     "train_f1": s.cv_results_["mean_train_score"][s.best_index_],
     "overfit_gap": s.cv_results_["mean_train_score"][s.best_index_] - s.best_score_,
     "best_params": s.best_params_}
    for name, s in searches.items()
]).sort_values("tuned_cv_f1", ascending=False).reset_index(drop=True)
tuned.round(4)

**Grid Search results**

- Tuning improved every tuned model. Random Forest gained the most: CV F1 rose from 0.660 to **0.687**. Limiting depth to 12 cut its overfitting gap from 0.34 to 0.14.
- HistGradientBoosting reached 0.673, SVM 0.672, Logistic Regression 0.671, and KNN 0.654.
- Logistic Regression and SVM barely overfit (gaps below 0.01) but have a lower F1 ceiling. The tree ensembles fit the training folds better and still give the highest CV F1.